# 09 — Machine Learning untuk Klasifikasi Citra

### Dari representasi piksel dan HOG menuju keputusan yang dapat diuji

**Seri:** Pengolahan Citra Digital dan *Scientific Imaging* · **Bahasa:** Indonesia · **Prasyarat:** Notebook 01–08, aljabar linear, peluang dasar, dan Python.

Pada delapan notebook pertama kita membangun pemahaman mengenai sensor, intensitas, filtering, gradien, segmentasi, frekuensi, warna dan fitur. Pertanyaan kini berubah: **bagaimana mengubah besaran terukur itu menjadi keputusan klasifikasi, tanpa keliru menyimpulkan bahwa skor pada satu dataset berlaku di semua kondisi akuisisi?**

Kita merumuskan klasifikasi sepuluh kelas, menurunkan model k-NN, regresi logistik multinomial dan *support vector machine* (SVM), membandingkan fitur piksel dan HOG pada data yang sama, lalu memisahkan **pemilihan model melalui validasi** dari **evaluasi sekali pada test**. Kita memeriksa probabilitas dan kalibrasi, ketidakseimbangan metrik, kesalahan individual, sensitivitas terhadap perubahan citra, serta ketidakpastian karena sampel uji yang terbatas.

**Studi kasus:** pengenalan angka tulisan tangan menggunakan `sklearn.datasets.load_digits`. Dataset sudah dibundel bersama scikit-learn: 1.797 citra 8×8 dan label 0–9. Angka intensitas 0–16 adalah level digital bawaan dataset, **bukan** fluks foton, radiansi, atau pengukuran warna yang terkalibrasi. Notebook mandiri, dapat dijalankan berurutan dan tidak memerlukan internet maupun GPU.

## Peta pembelajaran dan pertanyaan diagnostik

Setelah menuntaskan notebook ini, pembaca diharapkan mampu:

1. Menyatakan masalah klasifikasi citra sebagai pemetaan dari data pengukuran ke label dengan risiko generalisasi.
2. Menurunkan aturan keputusan Bayes, fungsi kerugian *cross-entropy*, jarak k-NN, dan gagasan margin SVM.
3. Membandingkan piksel dan HOG di bawah **split**, anggaran parameter, dan prosedur evaluasi yang konsisten.
4. Menjelaskan mengapa normalisasi harus di-*fit* pada data train saja dan bagaimana *pipeline* mencegah kebocoran statistik.
5. Memilih hiperparameter menggunakan validasi, lalu menjalankan test tanpa menggunakannya untuk memilih model ulang.
6. Menghitung confusion matrix, macro-F1, balanced accuracy, log loss, Brier score dan *expected calibration error* (ECE) secara tepat.
7. Mengevaluasi robustness pada citra test yang mengalami noise, rotasi, pergeseran dan blur; membedakan *stress test* dari data eksternal.
8. Menginterpretasikan interval kepercayaan bootstrap dan batas inferensi terhadap populasi penulis serta perangkat akuisisi baru.

**Diagnosis sebelum mulai:** Mengapa klasifikasi angka 8 dan 9 dapat tampak mudah setelah label diketahui tetapi ambigu pada contoh lain? Jika `StandardScaler` dilatih pada keseluruhan dataset sebelum pemisahan, apa yang bocor? Apakah `predict_proba = 0.99` berarti ada peluang 99% prediksi benar bagi setiap contoh?

**Aturan pembuktian:** semua angka metrik dihasilkan langsung dari eksekusi. Tidak ada akurasi yang ditetapkan sebelumnya, tidak ada pemilihan model menggunakan hasil test, dan tidak ada klaim bahwa dataset mewakili variasi tulisan tangan nyata di luar distribusi yang diamati.

## 1. Lingkungan dan kontrak reproduksibilitas

Seluruh eksperimen memakai `numpy`, `scipy`, `scikit-image`, `scikit-learn`, `pandas`, dan `matplotlib`. *Random seed* menetapkan pembagian data serta gangguan sintetis; versi pustaka dicatat karena algoritma optimisasi dapat berubah. Waktu komputasi hanya dipakai sebagai indikasi **lingkungan tempat notebook dijalankan**, bukan spesifikasi perangkat keras tertentu.

Kita akan memakai tiga kelompok data tetap: **train** untuk estimasi parameter, **validation** untuk memilih representasi/hyperparameter, dan **test** untuk mengukur performa akhir. Tambahan *stress test* dibuat dari citra test yang sama, sehingga bukan sampel baru yang independen.

In [ ]:
import platform, time, hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.feature import hog
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             log_loss, roc_auc_score)
from sklearn.base import clone

SEED=20261112
RNG=np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize':(9,4.5),'figure.dpi':105,'axes.grid':False})
print('Python',platform.python_version(),'scikit-learn',sklearn_version,
      'NumPy',np.__version__)

## 2. Apa yang dipelajari oleh classifier?

Sebuah citra berukuran $H\times W$ dan berkanal tunggal dapat ditulis sebagai vektor $\mathbf x\in\mathbb R^{HW}$. Model menghasilkan suatu label $\hat y=f_\theta(\mathbf x)$. Apabila fitur dipakai, langkahnya berubah menjadi

$$
\mathbf z=\phi(\mathbf x)\in\mathbb R^d,\qquad
\hat y=f_\theta(\mathbf z).
$$

Satu dataset terdiri atas $\mathcal D=\{(\mathbf x_i,y_i)\}_{i=1}^N$ yang diambil dari distribusi yang tidak sepenuhnya diketahui, $P_{\mathrm{data}}(X,Y)$. Pelatihan meminimalkan risiko empiris, tetapi tujuan sesungguhnya ialah risiko pada data baru:

$$
\widehat R(\theta)=\frac1{n_{\mathrm{train}}}\sum_{i\in\mathrm{train}}\ell(f_\theta(\mathbf x_i),y_i),
\quad R(\theta)=\mathbb E_{(X,Y)\sim P}[\ell(f_\theta(X),Y)].
$$

Perbedaan antara $\widehat R$ dan $R$ mencakup *generalization gap* dan ketidakpastian karena jumlah sampel terbatas. Algoritma yang sangat fleksibel dapat menghafal ciri train yang kebetulan muncul namun tidak stabil di luar data latihan.

**Hubungan dengan fisika pencitraan:** perubahan PSF, intensitas, noise, rotasi, atau skala mengubah $P(X\mid Y)$ dan dapat menurunkan performa walaupun identitas kelas $Y$ tidak berubah. Karena itu klasifikasi adalah masalah inferensi yang tidak terlepas dari proses akuisisi.

In [ ]:
# Dataset terbundel; tidak memerlukan file eksternal atau unduhan.
digits=load_digits()
X_images=digits.images.astype(np.float64)
X_pixels=digits.data.astype(np.float64)/16.0
y=digits.target.astype(int)
N=len(y); classes=np.arange(10)
assert X_images.shape==(1797,8,8) and X_pixels.shape==(1797,64)
print('Ukuran citra:',X_images.shape,'rentang level:',(X_images.min(),X_images.max()))
print(pd.DataFrame({'angka':classes,'jumlah':[np.sum(y==k) for k in classes]}).to_string(index=False))
fig,ax=plt.subplots(2,10,figsize=(12.2,3.1))
for k in classes:
    choices=np.flatnonzero(y==k)
    for r,j in enumerate(choices[:2]):
        ax[r,k].imshow(X_images[j],cmap='gray',vmin=0,vmax=16,interpolation='nearest')
        ax[r,k].set_title(str(k) if r==0 else '',fontsize=11);ax[r,k].axis('off')
fig.suptitle('Dua realisasi berbeda untuk setiap label angka');plt.tight_layout();plt.show()

### 2.1 Struktur informasi dalam citra 8×8

Ukuran citra yang kecil membuat klasifikasi cepat tetapi juga memunculkan persoalan spasial: ketebalan guratan, kemiringan, serta posisi relatif sangat memengaruhi label. Karena tidak disediakan ID penulis atau ID sesi akuisisi, dataset ini **tidak mendukung klaim evaluasi tingkat penulis (*writer-disjoint*)**. Pembagian acak terstratifikasi pada sampel menguji generalisasi pada observasi terpisah dari sumber dataset yang sama.

Citra ini juga bukan MNIST 28×28: metode HOG dan parameter selnya harus disesuaikan dengan kisi 8×8. Upaya membandingkan persentase akurasi langsung dengan publikasi MNIST tidak valid karena protokol serta distribusi berbeda.

In [ ]:
# Statistik dasar; komposisi kelas, konsentrasi intensitas, dan eksak duplikat.
intensity=np.mean(X_pixels,axis=1)
unique, multiplicity=np.unique(X_pixels,axis=0,return_counts=True)
print('Contoh citra duplikat eksak:',int((multiplicity-1).clip(min=0).sum()))
fig,ax=plt.subplots(1,2,figsize=(11,3.7))
ax[0].bar(classes,[np.sum(y==i) for i in classes]);ax[0].set_xticks(classes)
ax[0].set_title('Jumlah sampel per kelas');ax[0].set_xlabel('Label angka');ax[0].set_ylabel('Jumlah citra')
ax[1].boxplot([intensity[y==i] for i in classes],positions=classes,showfliers=False)
ax[1].set_title('Distribusi rerata intensitas per citra');ax[1].set_xlabel('Label angka');ax[1].set_ylabel('Intensitas digital ter-normalisasi')
plt.tight_layout();plt.show()

## 3. Membagi data dengan tujuan ilmiah yang jelas

*Stratification* mempertahankan proporsi kelas mendekati komposisi asli. Dua tahap pembagian membentuk train (sekitar 60%), validation (20%) dan test (20%) tanpa ada indeks yang berulang. Setiap sampel berada tepat pada satu kelompok. **Semua pemilihan fitur/model/parameter harus selesai menggunakan train dan validation sebelum test dibaca.**

Jika rekaman yang sama memiliki sejumlah foto berulang, perlu *group split*. Karena ID penulis/akuisisi tidak tersedia, split berbasis penulis tidak dapat dilakukan secara sah. Kita tetap memeriksa duplikat identik di antara kelompok dan mendokumentasikan potensi ketergantungan yang tak teramati. Pada penelitian nyata, data baru dari individu, batch atau perangkat yang sama tidak boleh otomatis dianggap independen.

In [ ]:
indices=np.arange(N)
idx_dev,idx_test=train_test_split(indices,test_size=.20,stratify=y,random_state=SEED)
idx_train,idx_val=train_test_split(idx_dev,test_size=.25,stratify=y[idx_dev],random_state=SEED+1)
assert len(set(idx_train)&set(idx_val))==len(set(idx_train)&set(idx_test))==len(set(idx_val)&set(idx_test))==0
assert len(idx_train)+len(idx_val)+len(idx_test)==N
splits={'train':idx_train,'validation':idx_val,'test':idx_test}
counts=pd.DataFrame({k:np.bincount(y[v],minlength=10) for k,v in splits.items()},index=classes)
counts.loc['Jumlah']=counts.sum(axis=0)
print(counts)
# Deteksi citra identik antar split (tanpa mengubah split atau melihat label untuk tuning).
def row_set(ix):return {X_images[j].tobytes() for j in ix}
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    print('Duplikat byte persis',a,'vs',b,':',len(row_set(splits[a])&row_set(splits[b])))

## 4. Baseline, akurasi dan keterbatasan kelas mayoritas

Sebelum menggunakan model rumit, kita menetapkan *baseline*. Classifier mayoritas selalu memprediksi kelas paling sering di train. Akurasinya hampir setara proporsi kelas mayoritas dan tidak memanfaatkan informasi citra. Sebuah model kompleks yang tak mengalahkan baseline mengindikasikan masalah serius.

Untuk $K$ kelas, confusion matrix $C$ memiliki elemen $C_{ij}$ = banyak contoh label benar $i$ yang diprediksi sebagai $j$. Akurasi total adalah $\sum_i C_{ii}/\sum_{ij}C_{ij}$. Akan tetapi kelas minoritas dapat nyaris diabaikan tanpa merusak akurasi apabila distribusi tidak seimbang. Oleh sebab itu kita juga menghitung *macro-F1* sebagai rata-rata harmonik precision–recall per kelas.

$$
P_k=\frac{TP_k}{TP_k+FP_k},\quad
R_k=\frac{TP_k}{TP_k+FN_k},\quad
F_{1,k}=\frac{2P_kR_k}{P_k+R_k},\quad
F_{1,\mathrm{macro}}=\frac1K\sum_{k=1}^K F_{1,k}.
$$

Jika suatu penyebut nol, perangkat lunak memerlukan aturan eksplisit; kita menetapkan `zero_division=0` untuk metrik laporan.

In [ ]:
dummy=DummyClassifier(strategy='most_frequent')
dummy.fit(X_pixels[idx_train],y[idx_train])
baseline_pred=dummy.predict(X_pixels[idx_val])
print('Baseline mayoritas — val accuracy:',round(accuracy_score(y[idx_val],baseline_pred),3),
      'macro-F1:',round(f1_score(y[idx_val],baseline_pred,average='macro',zero_division=0),3))
fig,ax=plt.subplots(figsize=(5.2,3.8))
ConfusionMatrixDisplay.from_predictions(y[idx_val],baseline_pred,display_labels=classes,
                                         cmap='Greys',ax=ax,colorbar=False)
ax.set_title('Confusion matrix baseline mayoritas (validasi)')
plt.tight_layout();plt.show()

## 5. Jarak sebagai hipotesis: k-nearest neighbors

Dalam k-NN, kelas sampel baru ditentukan oleh label tetangga dalam ruang fitur. Untuk jarak Euclidean,

$$
d(\mathbf z_i,\mathbf z_j)=\sqrt{\sum_{m=1}^d (z_{im}-z_{jm})^2}.
$$

Penggunaan jarak ini mengandung asumsi kuat: dimensi yang berbeda dapat dibandingkan secara berarti setelah skala yang cocok. Misalnya, satu fitur dengan rentang 10.000 bisa mendominasi fitur lain dengan rentang 0,01. *StandardScaler* mengestimasi $\mu_j$ dan $s_j$ menggunakan **train saja**, lalu menerapkan $z'_j=(z_j-\mu_j)/s_j$ ke validation/test.

Pada citra, k-NN sensitif terhadap translasi kecil: dua angka yang sama bisa bergeser satu piksel dan menghasilkan jarak Euclidean besar, sedangkan dua kelas berbeda bisa tampak mirip bila hanya dibandingkan intensitas piksel. Parameter $k$ mengatur bias–variance: terlalu kecil mudah mengikuti noise, terlalu besar bisa mengaburkan batas antarkelas.

In [ ]:
# Eksperimen jarak antar tiga gambar nyata; bukan ilustrasi bebas dari label.
idx0=np.flatnonzero(y==3)[:2]; idx1=np.flatnonzero(y==8)[:1]
chosen=np.r_[idx0,idx1]
D=np.linalg.norm(X_pixels[chosen,None,:]-X_pixels[None,chosen,:],axis=-1)
print('Label citra:',y[chosen].tolist())
print('Matriks jarak Euclidean pada vektor piksel:\n',np.round(D,3))
fig,ax=plt.subplots(1,3,figsize=(7.7,2.8))
for a,j in zip(ax,chosen):
    a.imshow(X_images[j],cmap='gray',vmin=0,vmax=16)
    a.set_title(f'Label {y[j]}');a.axis('off')
plt.tight_layout();plt.show()

## 6. Regresi logistik multinomial dan fungsi objektif

Untuk setiap kelas $k$, definisikan skor linear $a_k=\mathbf w_k^T\mathbf z+b_k$. Fungsi *softmax* mengubah skor menjadi probabilitas terhitung:

$$
p_\theta(Y=k\mid\mathbf z)=\frac{\exp(a_k)}{\sum_{j=0}^{K-1}\exp(a_j)}.
$$

Fungsi *cross-entropy* multikelas atau *negative log-likelihood* pada train ialah

$$
\mathcal L(\theta)= -\frac1n\sum_{i=1}^n\log p_\theta(y_i\mid\mathbf z_i)
+\frac\lambda2\|W\|_F^2.
$$

Suku pertama menghukum prediksi yang memberikan probabilitas kecil pada kelas benar. Penalti $L_2$ membatasi kompleksitas koefisien. Nilai `C` dalam scikit-learn mengatur kekuatan regularisasi secara terbalik: lebih besar $C$ berarti penalti relatif lebih lemah dalam konvensi solver terkait. **Prediksi probabilistik bukan jaminan kalibrasi**: model bisa sangat yakin namun salah, terutama di luar distribusi pelatihan.

Penyelesaian numerik melibatkan optimisasi convex untuk model linear dengan konfigurasi regularisasi standar, bukan optimisasi setiap piksel secara independen. Pada bagian berikutnya kita menunjukkan gradien softmax dan pemeriksaan numeriknya.

In [ ]:
# Gradient check untuk log-loss softmax pada satu sampel sintetik miniatur.
Z=np.array([[.2,-.5]],dtype=float)
y_one=np.array([2]);W=np.array([[.1,.4,-.3],[-.2,.2,.5]],dtype=float)
def stable_softmax(a):
    u=a-np.max(a,axis=1,keepdims=True)
    e=np.exp(u);return e/e.sum(axis=1,keepdims=True)
def ce_at(W_):
    p=stable_softmax(Z@W_)
    return -np.log(p[0,y_one[0]])
p=stable_softmax(Z@W)
Y=np.eye(3)[y_one]
grad=Z.T@(p-Y)
grad_fd=np.zeros_like(W)
eps=1e-6
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        perturb=np.zeros_like(W);perturb[i,j]=eps
        grad_fd[i,j]=(ce_at(W+perturb)-ce_at(W-perturb))/(2*eps)
assert np.allclose(grad,grad_fd,atol=1e-8)
print('Probabilitas kelas:',np.round(p.ravel(),4))
print('Maks |grad analitis − beda hingga|:',np.max(np.abs(grad-grad_fd)))

## 7. SVM: margin, kernel dan generalisasi

SVM linear membangun fungsi keputusan berdasarkan margin antara kelas. Untuk kasus biner, *soft-margin primal* dapat dirumuskan sebagai

$$
\min_{\mathbf w,b,\xi_i\ge0}\frac12\|\mathbf w\|_2^2+C\sum_i\xi_i,
\qquad y_i(\mathbf w^T\mathbf z_i+b)\ge1-\xi_i.
$$

Pada tugas sepuluh kelas, `SVC` memakai strategi *one-vs-one*, bukan persamaan biner tunggal ini secara langsung. Kernel RBF mengukur kedekatan nonlinear,

$$
K(\mathbf z_i,\mathbf z_j)=\exp\big(-\gamma\|\mathbf z_i-\mathbf z_j\|_2^2\big),\qquad\gamma>0.
$$

Parameter $C$ mengontrol penalti pelanggaran margin dan $\gamma$ mengontrol skala interaksi. Nilai ekstrem dapat membuat model terlalu kaku atau terlalu peka terhadap variasi train. `SVC.decision_function` **bukan** probabilitas terkalibrasi; jika diperlukan, kalibrasi harus dilakukan dengan prosedur yang tidak memanfaatkan test.

In [ ]:
# Visualisasi kernel RBF sebagai ukuran keserupaan satu dimensi.
delta=np.linspace(-4,4,501)
fig,ax=plt.subplots(figsize=(8.5,3.5))
for gamma in [0.1,0.5,2.0]:
    ax.plot(delta,np.exp(-gamma*delta**2),label=fr'$\gamma={gamma}$')
ax.set_xlabel('Selisih fitur dalam satu dimensi (unit standar)')
ax.set_ylabel('Nilai kernel K')
ax.set_title('Pengaruh gamma terhadap radius keserupaan RBF')
ax.legend();plt.tight_layout();plt.show()

## 8. Dari piksel ke HOG: apa yang berubah dalam representasi?

Vektor piksel 8×8 menyimpan semua 64 posisi intensitas digital, sedangkan *Histogram of Oriented Gradients* merangkum distribusi arah gradien lokal. Kita gunakan sel 4×4, blok 2×2 sel, dan 9 orientasi sehingga untuk satu citra 8×8 hanya terdapat satu blok HOG berdimensi $2\times2\times9=36$.

Hubungan dengan Notebook 08: HOG mengurangi sensitivitas terhadap pergeseran intensitas konstan dan menekankan struktur guratan. Namun pada citra 8×8, struktur sangat kasar: perubahan kecil posisi atau ketebalan garis dapat mengganti histogram gradien secara signifikan. Tidak ada alasan teoretis yang memastikan HOG selalu lebih baik daripada piksel mentah pada dataset ini.

Kedua representasi dibandingkan menggunakan split yang sama. HOG memiliki parameter **tetap yang dipilih sebelum melihat test**; jika kita mengubah sel atau jumlah orientasi setelah membaca test, itu termasuk *test-set overfitting* meskipun representasinya tidak dipelajari dengan gradient descent.

In [ ]:
# HOG deterministik; tidak mengestimasi statistik pada seluruh data.
def extract_hog_batch(image_arr):
    return np.stack([hog(im.astype(float)/16.0,orientations=9,
                         pixels_per_cell=(4,4),cells_per_block=(2,2),
                         block_norm='L2-Hys',feature_vector=True)
                     for im in image_arr])
T0=time.perf_counter()
X_hog=extract_hog_batch(X_images)
hog_seconds=time.perf_counter()-T0
assert X_hog.shape==(N,36)
print('Fitur piksel:',X_pixels.shape,'HOG:',X_hog.shape,
      'waktu ekstraksi HOG (mesin ini):',round(hog_seconds,3),'s')
fig,ax=plt.subplots(1,3,figsize=(9.5,3.1))
j=int(idx_train[0]);hog_vis=hog(X_images[j]/16.,orientations=9,pixels_per_cell=(4,4),
                               cells_per_block=(2,2),visualize=True,block_norm='L2-Hys')[1]
ax[0].imshow(X_images[j],cmap='gray',vmin=0,vmax=16);ax[0].set_title('Citra asli')
ax[1].imshow(hog_vis,cmap='magma');ax[1].set_title('Visualisasi HOG')
ax[2].bar(np.arange(36),X_hog[j]);ax[2].set_title('Vektor fitur HOG');ax[2].set_xlabel('Indeks fitur')
for a in ax[:2]:a.axis('off')
plt.tight_layout();plt.show()

## 9. Merancang pembandingan yang adil

Kita bandingkan baseline, k-NN, regresi logistik, SVM linear dan SVM RBF pada vektor piksel, lalu SVM RBF pada HOG. Untuk setiap kandidat, `StandardScaler`—bila digunakan—dibungkus sebagai bagian `Pipeline`; `fit()` dipanggil hanya pada **train**. Parameter kandidat didefinisikan sebelum melihat test dan dipilih berdasarkan **macro-F1 validasi**, dengan *balanced accuracy* dan nama konfigurasi sebagai pemecah nilai sama yang deterministik.

Tabel hasil validasi tidak boleh dibaca sebagai klaim bahwa satu arsitektur selalu unggul; tuning yang sangat terbatas tetap memilih konfigurasi yang kebetulan cocok dengan data validasi. Nilai test menyediakan satu pemeriksaan terpisah terhadap keputusan yang sudah dibuat.

**Definisi pembanding:** `Dummy` (mayoritas), k-NN piksel, regresi logistik piksel, SVM linear piksel, SVM RBF piksel, SVM RBF HOG. Tidak ada akses ke test saat memilih hyperparameter. Model HOG menggunakan ekstraksi deterministik yang telah dikunci pada subbab 8.

In [ ]:
# Kandidat ditetapkan dari teori, bukan dari pembacaan test.
FEATURES={'Piksel':X_pixels,'HOG':X_hog}
MODEL_SPECS=[]
def add(name,rep,estimator):MODEL_SPECS.append((name,rep,estimator))
add('Dummy','Piksel',DummyClassifier(strategy='most_frequent'))
for k in [3,7]:
    add(f'kNN-k{k}','Piksel',Pipeline([('scale',StandardScaler()),
                                        ('clf',KNeighborsClassifier(n_neighbors=k))]))
for c in [.2,2.0]:
    add(f'LogReg-C{c:g}','Piksel',Pipeline([('scale',StandardScaler()),
                     ('clf',LogisticRegression(C=c,max_iter=1000,solver='lbfgs'))]))
for c in [.3,3.0]:
    add(f'SVM-linear-C{c:g}','Piksel',Pipeline([('scale',StandardScaler()),
                     ('clf',SVC(kernel='linear',C=c))]))
for rep in ['Piksel','HOG']:
    for c in [1.0,10.0]:
        add(f'SVM-RBF-{rep}-C{c:g}',rep,Pipeline([('scale',StandardScaler()),
                     ('clf',SVC(kernel='rbf',C=c,gamma='scale'))]))
assert len(MODEL_SPECS)==11
print('Kandidat yang dideklarasikan sebelum test:',len(MODEL_SPECS))
for name,rep,_ in MODEL_SPECS:print('•',name,'/',rep)

### 9.1 Mengapa pipeline itu bagian dari validitas eksperimen?

Preprocessing merupakan operator yang mungkin memiliki parameter hasil estimasi. `StandardScaler` memerlukan nilai rerata dan deviasi baku; jika dihitung dari seluruh citra sebelum split, hasil train diam-diam memanfaatkan informasi distribusi validation/test. Hal serupa berlaku untuk PCA, seleksi fitur berdasarkan label, normalisasi per-batch, imputasi nilai hilang, dan kalibrasi probabilitas yang belajar dari label.

Secara formal, untuk fitur ke-$j$ rerata yang sah ialah

$$
\widehat\mu_{j,\mathrm{train}}=\frac1{n_{\mathrm{train}}}\sum_{i\in\mathrm{train}}x_{ij},
\qquad\widehat\sigma_{j,\mathrm{train}}=\sqrt{\frac1{n_{\mathrm{train}}}\sum_{i\in\mathrm{train}}(x_{ij}-\widehat\mu_j)^2}.
$$

Nilai tersebut kemudian dipakai untuk mentransformasi **semua** split, tanpa mengubahnya lagi. Apabila *pixel-level normalization* berupa pembagian dengan konstanta fisik atau level digital yang diketahui ($16$ pada dataset ini), operasi itu tidak mengestimasi statistik dari validation/test sehingga tidak setara dengan fitting `StandardScaler` secara global.

In [ ]:
VAL_ROWS=[];TRAINED={};PRED_VAL={};FIT_TIME={}
for name,rep,model in MODEL_SPECS:
    Z=FEATURES[rep]
    t=time.perf_counter()
    model.fit(Z[idx_train],y[idx_train])
    duration=time.perf_counter()-t
    pv=model.predict(Z[idx_val])
    TRAINED[name]=(model,rep)
    PRED_VAL[name]=pv
    FIT_TIME[name]=duration
    VAL_ROWS.append({'Model':name,'Fitur':rep,'F1 validasi':f1_score(y[idx_val],pv,average='macro',zero_division=0),
                     'Balanced acc validasi':balanced_accuracy_score(y[idx_val],pv),
                     'Akurasi validasi':accuracy_score(y[idx_val],pv),
                     'Fit (detik)':duration})
val_df=pd.DataFrame(VAL_ROWS).sort_values(['F1 validasi','Balanced acc validasi','Model'],
                                    ascending=[False,False,True]).reset_index(drop=True)
champion=val_df.iloc[0]['Model']
# Pembanding memakai keluarga algoritma berbeda untuk menghindari dua konfigurasi identik.
def algorithm_family(n):
    if n.startswith('SVM-linear'):return 'SVM linear'
    if n.startswith('SVM-RBF'):return 'SVM RBF'
    if n.startswith('kNN'):return 'kNN'
    if n.startswith('LogReg'):return 'Regresi logistik'
    return 'Dummy'
runner_up=next(n for n in val_df['Model'] if algorithm_family(n)!=algorithm_family(champion))
print('Model terpilih HANYA dari validasi:',champion)
print(val_df.round(4).to_string(index=False))

### 9.2 Apa arti perbedaan 0,5% pada validasi?

Jika dua model berbeda 0,005 pada macro-F1, belum tentu satu benar-benar lebih baik untuk populasi yang belum diamati. Sebuah nilai validasi dihitung dari sampel terbatas, dan pemilihan pemenang dari banyak kandidat menghasilkan *selection bias*. Di sini strategi kontrolnya sederhana: ruang pencarian kecil yang ditentukan sebelumnya, perbandingan pada split yang sama, serta test yang tidak disentuh sampai model terpilih. Penelitian lebih serius dapat memakai nested cross-validation, *multiple comparison correction*, dan test eksternal.

In [ ]:
# Lihat karakter model pada validasi saja.
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
order=val_df.sort_values('F1 validasi')
ax[0].barh(order['Model'],order['F1 validasi'])
ax[0].set_xlim(0,1.04);ax[0].set_xlabel('Macro-F1 validasi')
ax[0].set_title('Perbandingan pada data validasi')
ax[1].scatter(val_df['Fit (detik)']+.0001,val_df['F1 validasi'],s=70,alpha=.8)
for _,r in val_df.iterrows():
    if r['Model'] in [champion,runner_up]:
        ax[1].annotate(r['Model'],(r['Fit (detik)']+.0001,r['F1 validasi']),fontsize=8)
ax[1].set_xscale('log');ax[1].set_ylim(0,1.04)
ax[1].set_xlabel('Waktu fit kandidat (detik; skala log)')
ax[1].set_ylabel('Macro-F1 validasi');ax[1].set_title('Mutu vs biaya fit pada mesin ini')
plt.tight_layout();plt.show()

## 10. Diagnostik sebelum test: confusion matrix dan kelas sulit

Confusion matrix memperlihatkan arah kesalahan: prediksi label $j$ saat label sesungguhnya $i$ tidak sama dengan kesalahan sebaliknya. Untuk objek nyata, biaya salah klasifikasi sering asimetris; kesalahan mendeteksi cacat sebagai normal mungkin jauh lebih mahal daripada *false alarm*. Pada dataset angka, semua kelas diberi bobot yang sama sehingga macro-F1 sesuai untuk membandingkan perilaku antar kelas.

Kita melihat confusion matrix **validasi** untuk model terpilih, lalu melaporkan kelas dengan precision, recall dan F1 terendah. Analisis ini tidak mengubah hyperparameter lagi: ia berfungsi untuk menjelaskan pola kesalahan, bukan menyeleksi model melalui test.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.7))
ConfusionMatrixDisplay.from_predictions(y[idx_val],PRED_VAL[champion],
   display_labels=classes,cmap='Blues',ax=ax[0],colorbar=False)
ax[0].set_title(f'Validasi — {champion}')
val_cm=confusion_matrix(y[idx_val],PRED_VAL[champion],labels=classes)
row_norm=val_cm/val_cm.sum(axis=1,keepdims=True)
im=ax[1].imshow(row_norm,cmap='magma',vmin=0,vmax=1)
ax[1].set_xticks(classes);ax[1].set_yticks(classes)
ax[1].set_xlabel('Prediksi');ax[1].set_ylabel('Sebenarnya')
ax[1].set_title('Matriks dinormalisasi per kelas');plt.colorbar(im,ax=ax[1],shrink=.8)
plt.tight_layout();plt.show()
print(pd.DataFrame(classification_report(y[idx_val],PRED_VAL[champion],
   labels=classes,output_dict=True,zero_division=0)).T.round(3).head(10).to_string())

## 11. Learning curve: pengaruh jumlah sampel dan bias–variance

Akurasi train tinggi tidak otomatis menjelaskan kemampuan generalisasi. Model yang sangat fleksibel mungkin menghafal data train; model yang terlalu sederhana dapat gagal mengaproksimasi batas keputusan. Eksperimen berikut memperkecil train secara terstratifikasi, tetapi selalu mengevaluasi pada validation set yang sama.

**Perlu diingat:** *learning curve* ini bukan test akhir dan tidak dipakai untuk mencari banyak konfigurasi sampai memperoleh nilai maksimal. Kurva bertujuan mendiagnosis kebutuhan data. Satu titik kurva yang kebetulan naik/turun tidak otomatis membuktikan arah pengaruh ukuran sampel karena ada variabilitas pemilihan subset.

In [ ]:
# Kurva untuk model champion; ukuran subset dipilih sebelum eksperimen.
fractions=[.15,.30,.55,.80,1.0]
rows=[]
champ_est,champ_rep=TRAINED[champion]
Z=FEATURES[champ_rep]
for frac in fractions:
    if frac<1.:
        ix_small,_=train_test_split(idx_train,train_size=frac,stratify=y[idx_train],random_state=SEED+19)
    else:ix_small=idx_train
    tmp=clone(champ_est)
    tmp.fit(Z[ix_small],y[ix_small])
    rows.append({'Jumlah train':len(ix_small),'F1 train':f1_score(y[ix_small],tmp.predict(Z[ix_small]),average='macro'),
                 'F1 validasi':f1_score(y[idx_val],tmp.predict(Z[idx_val]),average='macro')})
learning=pd.DataFrame(rows)
print(learning.round(3).to_string(index=False))
fig,ax=plt.subplots(figsize=(8.1,3.9))
ax.plot(learning['Jumlah train'],learning['F1 train'],'o-',label='Train')
ax.plot(learning['Jumlah train'],learning['F1 validasi'],'s-',label='Validasi')
ax.set_ylim(0,1.03);ax.set_xlabel('Jumlah citra train')
ax.set_ylabel('Macro-F1');ax.set_title('Learning curve (validasi tetap)');ax.legend()
plt.tight_layout();plt.show()

## 12. Evaluasi akhir pada test yang belum digunakan

Setelah memutuskan `champion` hanya dari validasi, barulah kita membuka test. Semua model telah dilatih dengan konfigurasi kandidat yang ditentukan sebelumnya; menampilkan hasil test mereka untuk audit komparatif diperbolehkan, **tetapi hasil test tidak boleh digunakan untuk mengganti pemenang, menambah kandidat, atau mengubah preprocessing**. Jika ingin penelitian baru setelah melihat test, diperlukan test baru yang independen.

Untuk setiap kandidat kita laporkan macro-F1, balanced accuracy dan accuracy pada test. Kita lanjutkan analisis mendalam dengan model terpilih sebelumnya, bahkan jika model lain kebetulan mempunyai nilai test sedikit lebih tinggi.

In [ ]:
TEST_ROWS=[];PRED_TEST={};PRED_STRESS={}
for name,rep,_ in MODEL_SPECS:
    model,_=TRAINED[name]
    preds=model.predict(FEATURES[rep][idx_test])
    PRED_TEST[name]=preds
    TEST_ROWS.append({'Model':name,'Macro-F1 test':f1_score(y[idx_test],preds,average='macro',zero_division=0),
                     'Balanced acc test':balanced_accuracy_score(y[idx_test],preds),
                     'Accuracy test':accuracy_score(y[idx_test],preds)})
test_df=pd.DataFrame(TEST_ROWS)
comparison=val_df.merge(test_df,on='Model',validate='one_to_one')
print('Champion yang DIKUNCI dari validasi:',champion)
print(comparison[['Model','F1 validasi','Macro-F1 test','Balanced acc test','Fit (detik)']].round(4).to_string(index=False))
assert champion==val_df.iloc[0]['Model']

### 12.1 Confusion matrix test dan kesalahan individual

Confusion matrix test diperlakukan sebagai ringkasan error yang tersisa setelah model ditetapkan. Kita juga melihat contoh gambar yang salah prediksi, tetapi contoh tersebut tidak boleh diperlakukan sebagai sampel acak tak-bias jika dipilih karena menarik. Plot memberi **diagnosis mekanisme**, bukan validasi kausal mengenai gangguan citra.

In [ ]:
champ_pred=PRED_TEST[champion]
fig,ax=plt.subplots(1,2,figsize=(11.5,4.5))
ConfusionMatrixDisplay.from_predictions(y[idx_test],champ_pred,display_labels=classes,
                                        cmap='Blues',ax=ax[0],colorbar=False)
ax[0].set_title(f'Test final: {champion}')
per_class=classification_report(y[idx_test],champ_pred,labels=classes,
                                output_dict=True,zero_division=0)
f1cls=[per_class[str(k)]['f1-score'] for k in classes]
ax[1].bar(classes,f1cls);ax[1].set_xticks(classes);ax[1].set_ylim(0,1.05)
ax[1].set_xlabel('Label angka');ax[1].set_ylabel('F1 per kelas')
ax[1].set_title('Tidak semua kelas mempunyai kesulitan sama')
plt.tight_layout();plt.show()
wrong=idx_test[champ_pred!=y[idx_test]]
print('Jumlah kesalahan pada test:',len(wrong),'dari',len(idx_test))
fig,ax=plt.subplots(2,5,figsize=(10,4.6))
for a in ax.flat:a.axis('off')
for a,j in zip(ax.flat,wrong[:10]):
    pos=np.where(idx_test==j)[0][0]
    a.imshow(X_images[j],cmap='gray',vmin=0,vmax=16,interpolation='nearest')
    a.set_title(f'Benar {y[j]} → Pred {champ_pred[pos]}',fontsize=9);a.axis('off')
fig.suptitle('Contoh salah klasifikasi (dipilih karena error; bukan sampel acak)')
plt.tight_layout();plt.show()

## 13. Estimasi ketidakpastian metrik: bootstrap berpasangan

Perbandingan dua model pada **test yang sama** adalah *paired comparison*. Untuk dua model A dan B dengan prediksi $\hat y_A,\hat y_B$, kita mendefinisikan selisih estimasi macro-F1 sebagai

$$
\widehat\Delta=F_{1,\mathrm{macro}}(y,\hat y_A)-F_{1,\mathrm{macro}}(y,\hat y_B).
$$

Untuk bootstrap berpasangan, kita mengambil ulang indeks test yang sama bagi kedua model. Distribusi $\Delta^*$ memberi interval persentil empiris. Interval ini **bersyarat** pada model yang telah dilatih, *fixed test set*, dan asumsi observasi test dapat dipertukarkan. Jika terdapat korelasi antar tulisan dari orang yang sama, bootstrap tingkat foto akan terlalu optimistis; ID penulis tidak tersedia untuk memperbaikinya di sini.

Bahkan interval yang tidak mencakup nol tidak membuktikan keunggulan universal apabila model telah dicoba berkali-kali lalu dilaporkan berdasarkan test yang sama.

In [ ]:
# Bootstrap sampel test berpasangan untuk dua model yang dipilih dari VALIDASI.
b_rng=np.random.default_rng(SEED+32)
B=1200
ref_A=PRED_TEST[champion];ref_B=PRED_TEST[runner_up]
boot_deltas=np.empty(B)
for b in range(B):
    sel=b_rng.integers(0,len(idx_test),size=len(idx_test))
    boot_deltas[b]=(f1_score(y[idx_test][sel],ref_A[sel],average='macro',labels=classes,zero_division=0)
                    -f1_score(y[idx_test][sel],ref_B[sel],average='macro',labels=classes,zero_division=0))
delta=f1_score(y[idx_test],ref_A,average='macro')-f1_score(y[idx_test],ref_B,average='macro')
ci=np.quantile(boot_deltas,[.025,.975])
print(f'Δ macro-F1 test ({champion} − {runner_up}): {delta:+.4f}')
print('CI bootstrap 95% (dengan batasan yang dibahas):',np.round(ci,4))
fig,ax=plt.subplots(figsize=(8,3.6))
ax.hist(boot_deltas,bins=29,alpha=.8)
ax.axvline(0,color='black',linestyle=':',label='Δ = 0')
ax.axvline(delta,color='tab:red',label='Estimasi dari test')
ax.axvspan(*ci,color='tab:gray',alpha=.2,label='95% percentile CI')
ax.set_xlabel('Selisih macro-F1 (A−B)');ax.set_ylabel('Frekuensi bootstrap')
ax.set_title('Bootstrap berpasangan, unit = citra test');ax.legend()
plt.tight_layout();plt.show()

## 14. Ketidakpastian model: probabilitas, log loss, dan kalibrasi

Sebagian classifier menghasilkan skor yang dapat diubah menjadi probabilitas. **Probabilitas besar tidak sama dengan probabilitas yang akurat.** Model disebut terkalibrasi bila, secara agregat pada kondisi yang sesuai, prediksi dengan confidence sekitar 0,8 benar sekitar 80% dari waktu. Kita menguji model regresi logistik yang secara alami menghasilkan `predict_proba`, tanpa mengklaim bahwa softmax telah terkalibrasi.

Untuk sampel $(y_i,\mathbf p_i)$, log loss multikelas dan Brier score multikelas ialah

$$
\mathrm{NLL}=-\frac1n\sum_i\log p_{i,y_i},\qquad
\mathrm{Brier}=\frac1n\sum_i\sum_{k=0}^{K-1}(p_{ik}-\mathbb 1[y_i=k])^2.
$$

**ECE** dengan $M$ bin mengelompokkan nilai $c_i=\max_k p_{ik}$ dan membandingkan rata-rata confidence dengan proporsi prediksi benar:

$$
\mathrm{ECE}=\sum_{m=1}^M\frac{|B_m|}{n}\left|\mathrm{acc}(B_m)-\mathrm{conf}(B_m)\right|.
$$

ECE bergantung jumlah dan batas bin, serta bisa menutupi perilaku kelas tertentu. Plot berikut menggunakan model regresi logistik yang dipilih **hanya di antara konfigurasi regresi logistik berdasarkan validasi**, bukan sebagai pengganti champion setelah melihat test.

In [ ]:
# Pilih LogReg berdasarkan VALIDASI; laporan kalibrasi test bersifat diagnostik, bukan seleksi ulang.
logreg_name=val_df[val_df['Model'].str.startswith('LogReg')].iloc[0]['Model']
log_model,_=TRAINED[logreg_name]
probs=log_model.predict_proba(X_pixels[idx_test])
assert np.allclose(probs.sum(axis=1),1,atol=1e-12)
correct=(np.argmax(probs,axis=1)==y[idx_test]).astype(float)
confidence=probs.max(axis=1)
def reliability_bins(c,hit,n_bins=10):
    edges=np.linspace(0,1,n_bins+1);out=[]
    ids=np.minimum(np.digitize(c,edges[1:-1]),n_bins-1)
    for b in range(n_bins):
        m=(ids==b)
        if m.any():out.append((int(m.sum()),float(c[m].mean()),float(hit[m].mean())))
    return np.array(out,dtype=float)
r_bins=reliability_bins(confidence,correct,n_bins=10)
ece=np.sum((r_bins[:,0]/len(idx_test))*np.abs(r_bins[:,1]-r_bins[:,2]))
multibrier=np.mean(np.sum((probs-np.eye(10)[y[idx_test]])**2,axis=1))
print('Model probabilitas:',logreg_name)
print('Log loss:',round(log_loss(y[idx_test],probs,labels=classes),4),
      'Brier multikelas:',round(multibrier,4),'ECE 10 bin:',round(ece,4))
fig,ax=plt.subplots(1,2,figsize=(11.7,4.2))
ax[0].plot([0,1],[0,1],'k--',label='Kalibrasi sempurna')
ax[0].plot(r_bins[:,1],r_bins[:,2],'o-',label='Model')
for n_,c_,a_ in r_bins:ax[0].annotate(str(int(n_)),(c_,a_),xytext=(4,4),textcoords='offset points',fontsize=7)
ax[0].set_xlim(0,1);ax[0].set_ylim(0,1)
ax[0].set_xlabel('Confidence rata-rata');ax[0].set_ylabel('Proporsi benar')
ax[0].set_title('Reliability diagram; angka = jumlah sampel');ax[0].legend()
ax[1].hist(confidence,bins=np.linspace(0,1,21))
ax[1].set_xlabel('Confidence maksimum');ax[1].set_ylabel('Jumlah sampel')
ax[1].set_title('Distribusi confidence pada test')
plt.tight_layout();plt.show()

### 14.1 Contoh risiko: prediksi salah yang sangat yakin

Kesalahan dengan confidence besar merupakan contoh bahwa model dapat gagal dengan keyakinan tinggi. Namun jumlah *high-confidence errors* sangat bergantung definisi ambang serta representasi test yang diamati. Jika hanya sedikit error, tidak masuk akal menyimpulkan bahwa model telah aman terhadap semua citra ambiguitas.

Pada studi sistem nyata, prediksi dapat dikombinasikan dengan penolakan (*abstention*) ketika confidence rendah. Strategi demikian memerlukan evaluasi cakupan (*coverage*) dan risiko pada sampel yang diterima; ambang penolakan seharusnya dikunci menggunakan validation, bukan dipilih setelah memeriksa test.

In [ ]:
# Ambang absten telah ditentukan untuk demonstrasi; tidak dioptimasi pada test.
thresholds=[0.50,0.70,0.85,0.95]
triage=[]
for t in thresholds:
    accept=confidence>=t
    risk=np.nan if not accept.any() else 1-correct[accept].mean()
    triage.append({'Ambang confidence':t,'Coverage':accept.mean(),
                   'Error diterima':risk,'Jumlah diterima':int(accept.sum())})
print(pd.DataFrame(triage).round(3).to_string(index=False))
errors=np.flatnonzero((correct==0)&(confidence>=0.8))
fig,ax=plt.subplots(1,5,figsize=(11,2.4))
for a in ax:a.axis('off')
for a,j in zip(ax,errors[:5]):
    a.imshow(X_images[idx_test[j]],cmap='gray',vmin=0,vmax=16)
    a.set_title(f'{y[idx_test[j]]} → {np.argmax(probs[j])}\nconf={confidence[j]:.2f}',fontsize=8)
plt.suptitle('Salah dengan confidence ≥ 0,8 (jika ada)');plt.tight_layout();plt.show()

## 15. Domain shift: ketika proses pembentukan citra berubah

Sebagian gambar baru mungkin diambil dengan kamera yang lebih kabur, posisi objek bergeser, atau noise meningkat. Kita membuat variasi terkontrol dari citra **test** yang telah disisihkan: noise Gaussian tambahan, rotasi, translasi dan blur. Gangguan ini **bukan** data lapangan dari perangkat berbeda; ia menguji stabilitas model terhadap keluarga transformasi yang sengaja kita tetapkan.

Misalkan operator degradasi $T_\eta$ mengubah $\mathbf x$ menjadi $\mathbf x'=T_\eta(\mathbf x)$. Pada asumsi *label-preserving perturbation*, label seharusnya sama: $Y'=Y$. Namun rotasi/blur ekstrem bisa mengubah angka secara semantik (misalnya 6 terhadap 9), sehingga label-preserving tidak otomatis benar untuk semua $\eta$. Pada percobaan ini amplitudo gangguan dipilih sedang, contoh visual tetap perlu diperiksa, dan interpretasi harus memperhitungkan kemungkinan perubahan keterbacaan manusia.

**Protokol:** parameter gangguan ditentukan sebelum melihat skor stress, tidak dipilih berdasarkan hasilnya. Data yang dipakai tetap indeks test yang sama; kita membandingkan performa secara berpasangan, bukan menyatakan telah menguji populasi penulis baru.

In [ ]:
# Stres citra: semua sumber berasal dari test yang sama; label diwariskan secara bersyarat.
base_test=X_images[idx_test]/16.0
r_stress=np.random.default_rng(SEED+404)
STRESS_IMAGES={'Asli':base_test.copy()}
STRESS_IMAGES['Noise']=np.clip(base_test+r_stress.normal(0,0.17,size=base_test.shape),0,1)
STRESS_IMAGES['Blur']=np.stack([ndi.gaussian_filter(im,sigma=.65,mode='constant') for im in base_test])
STRESS_IMAGES['Rotasi']=np.stack([ndi.rotate(im,angle=(-10 if j%2 else 10),
                                            reshape=False,order=1,mode='constant',cval=0)
                                  for j,im in enumerate(base_test)])
STRESS_IMAGES['Translasi']=np.stack([ndi.shift(im,shift=(.70,-.60),order=1,
                                              mode='constant',cval=0) for im in base_test])
STRESS_IMAGES['Gabungan']=np.stack([
    ndi.gaussian_filter(ndi.rotate(im,angle=(7 if j%2 else -7),
         reshape=False,order=1,mode='constant',cval=0),sigma=.4,mode='constant')
    for j,im in enumerate(STRESS_IMAGES['Noise'])])
assert all(a.shape==(len(idx_test),8,8) for a in STRESS_IMAGES.values())
fig,ax=plt.subplots(3,len(STRESS_IMAGES),figsize=(12.5,5.7))
for c,(name,imgs) in enumerate(STRESS_IMAGES.items()):
    for r in range(3):
        ax[r,c].imshow(imgs[r],cmap='gray',vmin=0,vmax=1,interpolation='nearest')
        ax[r,c].set_title(name if r==0 else '',fontsize=9);ax[r,c].axis('off')
plt.suptitle('Citra test yang sama pada enam kondisi (label diasumsikan tetap)')
plt.tight_layout();plt.show()

### 15.1 Membandingkan robustness lintas representasi

Semua model yang sebelumnya dilatih dievaluasi pada semua kondisi gangguan, tanpa melakukan *fit* ulang. Dengan demikian perbedaan antara kondisi normal dan stress disebabkan transformasi input, bukan perubahan training model. Untuk HOG, ekstraksi dilakukan kembali pada citra yang terdegradasi menggunakan definisi fitur sama persis. Untuk piksel, citra 8×8 diratakan menjadi 64 dimensi.

Kita laporkan macro-F1 pada setiap kondisi dan agregat perubahan dari kondisi asli. Rata-rata beberapa kondisi stress dipakai hanya sebagai ringkasan deskriptif; tidak merupakan estimasi distribusi gangguan dunia nyata.

In [ ]:
STRESS_FEATURES={}
for name,im in STRESS_IMAGES.items():
    STRESS_FEATURES[(name,'Piksel')]=im.reshape(len(im),-1)
    STRESS_FEATURES[(name,'HOG')]=extract_hog_batch(im*16.0)
stress_rows=[]
for name,rep,_ in MODEL_SPECS:
    model,_=TRAINED[name]
    for condition in STRESS_IMAGES:
        pp=model.predict(STRESS_FEATURES[(condition,rep)])
        stress_rows.append({'Model':name,'Kondisi':condition,
                            'Macro-F1':f1_score(y[idx_test],pp,average='macro',zero_division=0),
                            'Accuracy':accuracy_score(y[idx_test],pp)})
stress_df=pd.DataFrame(stress_rows)
heat=stress_df.pivot(index='Model',columns='Kondisi',values='Macro-F1')
heat=heat.loc[val_df['Model'],list(STRESS_IMAGES)]
print(heat.round(3).to_string())
fig,ax=plt.subplots(figsize=(12.5,5.6))
im=ax.imshow(heat.values,cmap='viridis',vmin=0,vmax=1,aspect='auto')
ax.set_xticks(np.arange(len(heat.columns)),heat.columns)
ax.set_yticks(np.arange(len(heat.index)),heat.index)
ax.set_title('Macro-F1 pada test normal dan perturbasi terkontrol')
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        ax.text(j,i,f'{heat.iat[i,j]:.2f}',ha='center',va='center',fontsize=8,
                color='white' if heat.iat[i,j]<.5 else 'black')
fig.colorbar(im,ax=ax,label='Macro-F1',shrink=.88)
plt.tight_layout();plt.show()

## 16. Diagnostik perpindahan domain: jenis kegagalan yang berubah

Kita mengamati perubahan kelas yang paling sering salah untuk model terpilih pada kondisi gabungan, dibandingkan test normal. Jika transformasi mengaburkan lingkaran digit 8 atau menggeser bagian angka mendekati batas 8×8, model mungkin mengalami perubahan prediksi. Namun angka kesalahan tidak otomatis menunjukkan fitur fisik yang menjadi sebab, sebab transformasi gabungan sekaligus mengubah beberapa faktor.

Dalam studi instrumentasi, pemisahan penyebab menuntut eksperimen *one-factor-at-a-time* atau rancangan faktorial dengan pengulangan. Simulasi domain shift hanya awal: pengujian berbeda perangkat/batch/lokasi diperlukan untuk klaim generalisasi lapangan.

In [ ]:
chosen_model,chosen_rep=TRAINED[champion]
pp_stress=chosen_model.predict(STRESS_FEATURES[('Gabungan',chosen_rep)])
fig,ax=plt.subplots(1,2,figsize=(11.5,4.6))
ConfusionMatrixDisplay.from_predictions(y[idx_test],champ_pred,display_labels=classes,
                                        cmap='Blues',ax=ax[0],colorbar=False)
ax[0].set_title('Test normal')
ConfusionMatrixDisplay.from_predictions(y[idx_test],pp_stress,display_labels=classes,
                                        cmap='Oranges',ax=ax[1],colorbar=False)
ax[1].set_title('Test stress gabungan')
plt.tight_layout();plt.show()
changed=np.flatnonzero((champ_pred==y[idx_test])&(pp_stress!=y[idx_test]))
fig,ax=plt.subplots(2,6,figsize=(12.2,4.6))
for a in ax.flat:a.axis('off')
for col,j in enumerate(changed[:6]):
    ax[0,col].imshow(base_test[j],cmap='gray',vmin=0,vmax=1)
    ax[1,col].imshow(STRESS_IMAGES['Gabungan'][j],cmap='gray',vmin=0,vmax=1)
    ax[0,col].set_title(f'Benar {y[idx_test[j]]}',fontsize=9)
    ax[1,col].set_title(f'Pred {pp_stress[j]}',fontsize=9)
plt.suptitle('Benar pada normal, salah setelah degradasi (bila ada)')
plt.tight_layout();plt.show()
print('Prediksi berubah dari benar menjadi salah:',len(changed))

## 17. Analisis berulang tanpa menyentuh test final

Kita tidak dapat menyimpulkan *model stability* dari satu split validasi. Untuk memperoleh gambaran variabilitas karena pembagian data, jalankan *repeated stratified k-fold* pada **train saja** menggunakan dua konfigurasi yang sudah ditetapkan sebelumnya. Fitting `StandardScaler` dan classifier terjadi dalam setiap lipatan; data test utama tetap disimpan.

Evaluasi berulang ini bersifat **diagnostik**: skor dari fold yang saling tumpang tindih tidak independen dan tidak boleh diinterpretasikan sebagai enam eksperimen dari populasi baru. Ia juga tidak menggantikan nested cross-validation bila tuning model meluas. Catat jumlah pengulangan, seed, dan metrik, serta jangan mengubah champion berdasarkan data test.

In [ ]:
cv=RepeatedStratifiedKFold(n_splits=3,n_repeats=2,random_state=SEED+92)
cvrows=[]
for name in [champion,runner_up]:
    model,rep=TRAINED[name]
    Z=FEATURES[rep][idx_train]
    scores=cross_validate(clone(model),Z,y[idx_train],cv=cv,scoring='f1_macro',n_jobs=1)
    for i,score in enumerate(scores['test_score']):
        cvrows.append({'Model':name,'Fold/ulang':i+1,'F1':score})
cvdf=pd.DataFrame(cvrows)
print(cvdf.groupby('Model')['F1'].agg(['mean','std','min','max']).round(4).to_string())
fig,ax=plt.subplots(figsize=(8.5,3.8))
for name,subset in cvdf.groupby('Model'):
    ax.plot(subset['Fold/ulang'],subset['F1'],'o-',label=name)
ax.set_xlabel('Indeks fold pada repeated 3-fold CV (train only)')
ax.set_ylabel('Macro-F1');ax.set_ylim(0,1.05)
ax.set_title('Variabilitas akibat pembagian data pada train');ax.legend()
plt.tight_layout();plt.show()

## 18. Eksperimen ablation: memisahkan efek fitur dari model

Dalam membandingkan piksel versus HOG, perubahan representasi harus dibedakan dari perubahan jenis classifier. Salah satu *ablation* yang adil adalah membandingkan **SVM RBF dengan nilai C yang sama** pada dua representasi berbeda, memakai split yang identik. Ini sudah tersedia dari kandidat yang kita latih. Kita dapat membandingkan **hasil validasi** untuk semua pasangan yang sesuai, serta menghitung dimensi fitur sebagai proksi kasar ukuran input dan waktu prediksi pada mesin uji.

HOG lebih ringkas, tetapi biaya ekstraksi HOG tidak nol; penghematan dimensi input belum tentu membuat pipeline lengkap lebih cepat. SVM RBF juga memiliki biaya prediksi yang bergantung jumlah *support vector*. Kita laporkan biaya ekstraksi dan prediksi secara terpisah tanpa mengklaim satu metode optimal secara universal.

In [ ]:
# Ablation berdasarkan hasil VALIDASI yang sudah dihitung, tanpa training baru pada test.
abl=val_df[val_df['Model'].str.contains('SVM-RBF')].copy()
abl['Dimensi input']=abl['Fitur'].map({'Piksel':64,'HOG':36})
print(abl[['Model','Dimensi input','F1 validasi','Fit (detik)']].round(4).to_string(index=False))
timing_rows=[]
for name,rep,_ in MODEL_SPECS:
    if not name.startswith('SVM-RBF'):continue
    model,_=TRAINED[name]
    ZZ=FEATURES[rep][idx_val]
    t0=time.perf_counter()
    for i in range(3):model.predict(ZZ)
    ms=(time.perf_counter()-t0)*1000/(3*len(ZZ))
    timing_rows.append({'Model':name,'Inferensi ms/citra (fitur sudah ada)':ms})
pred_time_df=pd.DataFrame(timing_rows)
print(pred_time_df.round(4).to_string(index=False))
print('Ekstraksi HOG dari 1797 citra:',round(hog_seconds,3),'detik pada mesin ini')

## 19. Mengapa akurasi tidak sama dengan validitas ilmiah?

Ada tiga jenis kesimpulan yang perlu dipisahkan:

1. **Kinerja komputasi pada dataset:** hasil macro-F1 dan confusion matrix setelah protokol split tertentu.
2. **Ketahanan terhadap gangguan sintetis:** perubahan hasil ketika citra test diubah dengan operator gangguan yang ditentukan.
3. **Generalisasi perangkat/manusia/lingkungan:** kinerja pada distribusi berbeda yang benar-benar dikumpulkan dari sumber baru.

Eksperimen ini hanya menyajikan (1) dan (2). Tidak ada ID penulis, eksperimen multisensor, kalibrasi radiometrik kamera, ataupun ground truth tugas industri. Maka angka yang dihasilkan tidak membuktikan tingkat keamanan, keadilan antar kelompok, atau kelayakan otomatisasi keputusan penting.

Di sisi lain, desain ini tetap menghasilkan pelajaran metodologis nyata: keputusan validasi dikunci sebelum test, fitur dan model dibandingkan pada data yang sama, ketidakpastian diungkapkan, serta dampak degradasi dapat diukur langsung.

In [ ]:
# Ringkasan kuantitatif: angka diperoleh dari hasil kode, bukan ditulis manual.
row=next(r for r in TEST_ROWS if r['Model']==champion)
normal=float(heat.loc[champion,'Asli']); combined=float(heat.loc[champion,'Gabungan'])
print('=== RINGKASAN EKSPERIMEN NOTEBOOK 09 ===')
print('Jumlah sampel:',N,'/ train',len(idx_train),'/ validation',len(idx_val),'/ test',len(idx_test))
print('Model terpilih dari validasi:',champion)
print('F1 test normal:',round(row['Macro-F1 test'],4))
print('F1 test gabungan:',round(combined,4))
print('Penurunan absolut F1:',round(normal-combined,4))
print('95% CI bootstrap selisih pemenang vs pembanding beda keluarga:',np.round(ci,4).tolist())
print('Catatan: validitas hanya pada sumber data dan gangguan yang didefinisikan.')

## 20. Pemeriksaan numerik dan integritas data

Notebook yang bisa berjalan belum tentu benar. Pemeriksaan berikut menilai kontrak struktur, label, split, *fit* statistik yang sah, dimensi fitur, jumlah peluang softmax, prediksi valid, serta keterpisahan sumber test dari train. Pemeriksaan ini menangkap beberapa kesalahan implementasi tetapi **bukan pembuktian bahwa model ilmiah atau protokol akuisisi sepenuhnya valid**.

In [ ]:
# Pemeriksaan programatik akhir, tanpa membandingkan hasil dengan angka buatan.
assert X_images.shape==(1797,8,8)
assert X_pixels.shape==(1797,64) and X_hog.shape==(1797,36)
assert set(np.unique(y))==set(classes)
assert (X_pixels>=0).all() and (X_pixels<=1).all()
assert len(set(idx_train)&set(idx_test))==0 and len(set(idx_val)&set(idx_test))==0
assert len(set(idx_train)&set(idx_val))==0
assert all(set(row_set(splits[a])).isdisjoint(row_set(splits[b]))
  for a,b in [('train','validation'),('train','test'),('validation','test')])
assert all(np.isfinite(r['Macro-F1 test']) and 0<=r['Macro-F1 test']<=1 for r in TEST_ROWS)
assert set(TEST_ROWS[0])=={'Model','Macro-F1 test','Balanced acc test','Accuracy test'}
assert np.isfinite(boot_deltas).all() and ci[0]<=ci[1]
assert np.allclose(probs.sum(axis=1),1)
assert set(stress_df.Kondisi.unique())==set(STRESS_IMAGES)
assert champion in TRAINED and champion in PRED_TEST and champion in PRED_VAL
# Distribusi standar pada train menjadi mean 0 (kolom ber-variansi nol tetap nol).
scaler=StandardScaler().fit(X_pixels[idx_train]);ztr=scaler.transform(X_pixels[idx_train])
assert np.max(np.abs(ztr.mean(axis=0)))<1e-12
print('LULUS: integritas indeks, tidak ada duplikat eksak lintas split, bentuk fitur, metrik, probabilitas, dan sintaks pengujian.')

## 21. Praktikum, latihan analitis, dan HOTS

**Alur praktikum yang direkomendasikan:** (a) eksplorasi citra dan anotasi keterbatasan dataset; (b) bangun split yang dibekukan; (c) estimasi baseline dan bandingkan piksel-HOG; (d) pilih satu model pada validation; (e) uji sekali dan audit confusion matrix; (f) lakukan stress terkontrol; (g) tulis hasil, uncertainty dan batas klaim.

**Latihan konseptual**

1. Jika matriks confusion kelas 3 menghasilkan $TP=30$, $FP=5$, dan $FN=10$, hitung precision, recall dan F1 serta jelaskan apakah akurasi global dapat ditentukan dari informasi itu saja.
2. Untuk logits $(1,2,3)$, hitung probabilitas softmax secara manual. Bandingkan dengan logits $(101,102,103)$ dan jelaskan pentingnya stabilisasi numerik.
3. Untuk dua vektor fitur 2D $\mathbf z_1=(1,10)$, $\mathbf z_2=(2,11)$, hitung jarak Euclidean sebelum dan setelah satuan dimensi pertama dikalikan 100. Interpretasikan peran `StandardScaler`.

**Latihan analitis**

4. Turunkan gradien cross-entropy softmax terhadap parameter linear dan cocokkan dengan pemeriksaan beda-hingga di kode.
5. Bandingkan macro-F1 dan weighted-F1 ketika satu kelas hanya 1% dari data. Jelaskan metrik apa yang lebih relevan bagi deteksi kejadian langka.
6. Ubah ukuran sel HOG secara *pre-registered* sebelum evaluasi test baru. Diskusikan keterbatasan dimensi yang hanya 8×8.
7. Lakukan eksperimen *learning curve* dengan lima subset train dan tiga seed. Laporkan variasi yang berasal dari pemilihan subset, bukan hanya kurva rata-rata.

**Latihan HOTS**

8. Tersedia 20 pasien dengan lima citra mikroskop per pasien. Rancang pembagian train–validation–test yang mencegah *patient leakage*. Jelaskan unit resampling bagi bootstrap dan generalisasi populasi.
9. Model A mempunyai macro-F1 0,97 tetapi ECE 0,18; model B macro-F1 0,95 dan ECE 0,02. Dalam sistem yang mengizinkan abstention dan biaya error asimetris, usulkan kriteria pemilihan dan protokol validasi; jangan hanya memilih angka terbesar.
10. Pada dataset baru, blur mengubah label persepsi operator untuk beberapa angka. Rancang audit apakah gangguan benar-benar *label-preserving* sebelum menggunakan kurva robustness.
11. Jelaskan bagaimana pemakaian test set untuk menentukan ambang confidence menimbulkan bias. Rancang subset kalibrasi yang independen atau skema cross-validation dari development set.
12. Jika dataset diambil dari dua kamera dengan pola noise berbeda, bedakan efek *covariate shift*, kemungkinan *label shift*, dan risiko model menggunakan kamera sebagai *shortcut*.

**Produk akhir praktikum:** laporan teknis 5–7 halaman dalam Bahasa Indonesia dengan pertanyaan ilmiah, asumsi data, diagram pipeline, persamaan, tabel perbandingan, confusion matrix, kalibrasi, stress test, interval ketidakpastian, dokumentasi reproduksi, dan batas kesimpulan. Lampirkan tabel split dan kode yang dapat dijalankan.

## 22. Rangkuman dan koneksi antarbab

- Model supervised mengaproksimasi hubungan antara citra dan label dari contoh beranotasi; performa luar-sampel tidak dijamin oleh minimisasi loss train.
- Piksel dan HOG menyimpan informasi yang berbeda; pilihan fitur harus diuji secara adil pada data yang sama.
- k-NN memerlukan asumsi metrik, regresi logistik mengoptimasi likelihood teratur, dan SVM memanfaatkan margin serta kernel.
- Data leakage dapat terjadi pada preprocessing, tuning dan kalibrasi, bukan hanya pada pelatihan classifier.
- Macro-F1 menangkap keseimbangan kelas, tetapi tidak menggantikan confusion matrix dan analisis biaya kesalahan.
- Probabilitas tinggi belum tentu terkalibrasi; confidence harus diperiksa terhadap frekuensi kebenaran pada data yang relevan.
- *Stress test* sintetis memberi bukti sensitivitas terhadap gangguan tertentu, bukan validasi terhadap kamera, penulis, atau populasi baru.
- Ketidakpastian bootstrap bersifat kondisional pada unit sampel yang dipakai; jika ada pengelompokan tak teramati, CI dapat menyesatkan.

**Koneksi berikutnya:** pembelajaran mendalam mengembangkan ekstraksi fitur yang dipelajari langsung dari data, namun tetap memerlukan pembandingan yang adil, *ablation*, kalibrasi, dan evaluasi ketahanan. Pada inverse problems dan Bayesian imaging, pengaruh noise serta ketidakpastian model akan dipelajari lebih formal; pada domain shift dan desain eksperimen, kita akan menguji apakah hubungan input–label bertahan ketika sumber data dan protokol akuisisi berubah. Materi ini mengikat representasi citra pada Notebook 01–08 dengan **disiplin inferensi** yang diperlukan sebelum menyatakan hasil sebuah model dapat dipercaya.

## Referensi dan bacaan lanjut

1. Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. Bab 3–4: model linear dan klasifikasi.
2. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning*, edisi ke-2. Springer. DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).
3. Cortes, C., & Vapnik, V. (1995). *Support-Vector Networks*. Machine Learning, 20, 273–297. DOI: [10.1007/BF00994018](https://doi.org/10.1007/BF00994018).
4. Dalal, N., & Triggs, B. (2005). *Histograms of Oriented Gradients for Human Detection*. IEEE CVPR. DOI: [10.1109/CVPR.2005.177](https://doi.org/10.1109/CVPR.2005.177).
5. Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). *On Calibration of Modern Neural Networks*. ICML. [PMLR 70:1321–1330](https://proceedings.mlr.press/v70/guo17a.html).
6. Dokumentasi scikit-learn: [Digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), [Pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline), [Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html), dan [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html).
7. Dokumentasi scikit-image: [Histogram of Oriented Gradients](https://scikit-image.org/docs/stable/auto_examples/features_detection/plot_hog.html).

**Provenance dan etika:** gambar berasal dari dataset `Digits` yang didistribusikan di scikit-learn; notebook hanya membutuhkan instalasi scikit-learn dan tidak menyertakan ulang gambar dalam repository. Identitas penulis, perangkat akuisisi, dan protokol sensor tidak disediakan oleh antarmuka dataset; tidak boleh disimpulkan dari gambar.